# RQ1 Evaluation: Translation Accuracy of LLM-Generated Constraints

This notebook evaluates **Research Question 1 (RQ1)**:
> **RQ1 (Translation Accuracy):** How accurately can the Large Language Model parse complex, abstract, and potentially ambiguous natural language directives from the user and translate them into mathematically precise Must-link and Cannot-link constraints?

We employ a **Hybrid Evaluation Framework**:
1. **Strict Quantitative Metrics (Precision, Recall, F1):** Used for *Level 1 (Simple/Direct)* directives, comparing LLM-generated constraints directly against the ground-truth ArXiv category groupings.
2. **LLM-as-a-Judge Semantic Accuracy:** Used for *all levels*, utilizing a separate LLM (e.g., GPT-4o) to evaluate whether each individual generated constraint makes logical and semantic sense under the user's natural language directive.

### Step 0: Set Up Environment & Configuration

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

# Suppress warnings from third-party libraries for clean outputs
warnings.filterwarnings("ignore", category=UserWarning, module="metric_learn")
warnings.filterwarnings("ignore", category=FutureWarning)

# Add source directory to python path
sys.path.append(os.path.abspath('..'))

from src.dataset.loader import load_dataset
from src.agent.cloud_agent import OpenAIAgent, GeminiAgent, GitHubModelsAgent
from src.agent.local_agent import OllamaLocalAgent
from src.evaluation.metrics import compute_ari, compute_nmi

print("Environment and imports set up successfully!")

### Step 1: Load Data & Select Evaluation Sample

In [ ]:
# Load dataset
dataset_path = "../data/arxiv_processed.json"
if not os.path.exists(dataset_path):
    dataset_path = "../data/arxiv_processed_sample.json"

dataset = load_dataset(dataset_path)
texts = dataset.get_texts()
categories = dataset.get_aspect_labels("category")

print(f"Loaded {len(texts)} documents.")

# Select a fixed set of 20 documents for evaluation (using deterministic random seed)
np.random.seed(42)
sampled_indices = sorted(list(np.random.choice(len(texts), 20, replace=False)))
sampled_texts = [texts[idx] for idx in sampled_indices]
sampled_categories = [categories[idx] for idx in sampled_indices]

print(f"Sampled 20 documents indices: {sampled_indices}")
print("\nSampled documents preview:")
for idx, (text, cat) in enumerate(zip(sampled_texts, sampled_categories)):
    print(f"[{idx:2d}] (Category: {cat:8s}) {text[:90]}...")

### Step 2: Define Predefined Test Directives Suite

In [ ]:
# Predefined directives suite categorized by complexity/abstraction level
directives_suite = {
    "Level 1: Simple / Direct (Topic-based)": [
        "Group documents by their main academic discipline (e.g., Physics versus Math/Computer Science).",
        "Separate physics papers from mathematics and computer science papers.",
        "Cluster papers into theoretical physics (e.g. hep-ph) versus mathematics/computer science."
    ],
    "Level 2: Abstract / High-level (Methodology & Focus)": [
        "Group the documents by style of methodology: separate papers that focus on numerical computations/simulations from those that are purely mathematical proofs and derivations.",
        "Cluster papers based on whether they describe concrete physical systems (like particles, stars, Earth) versus abstract mathematical/algorithmic systems.",
        "Separate papers presenting software/algorithmic solutions from those proposing physical theories/phenomenology."
    ],
    "Level 3: Ambiguous / Subjective (Vague)": [
        "Group papers that feel like they study microscopic things versus macroscopic or cosmic things.",
        "Cluster papers by how 'applied' they feel to real-world engineering vs purely academic research.",
        "Group papers depending on whether they study dynamics/evolution over time versus static structures."
    ]
}

print(f"Test suite defined with {sum(len(v) for v in directives_suite.values())} directives across 3 levels.")

### Step 3: Configure Generator & Judge Agents

In [ ]:
# Generator Agent (translates directives into constraints)
generator_agent = None
if os.getenv("GITHUB_TOKEN"):
    generator_agent = GitHubModelsAgent(model_name="gpt-4o-mini", verbose=False)
elif os.getenv("OPENAI_API_KEY"):
    generator_agent = OpenAIAgent(model_name="gpt-4o-mini", verbose=False)
elif os.getenv("GEMINI_API_KEY"):
    generator_agent = GeminiAgent(model_name="gemini-1.5-flash", verbose=False)
else:
    generator_agent = OllamaLocalAgent(base_url="http://localhost:11434", model_name="qwen2.5:7b", verbose=False)

# Judge Agent (evaluates constraint accuracy)
judge_agent = None
if os.getenv("GITHUB_TOKEN"):
    judge_agent = GitHubModelsAgent(model_name="gpt-4o", verbose=False)
elif os.getenv("OPENAI_API_KEY"):
    judge_agent = OpenAIAgent(model_name="gpt-4o", verbose=False)
elif os.getenv("GEMINI_API_KEY"):
    judge_agent = GeminiAgent(model_name="gemini-1.5-pro", verbose=False)
else:
    judge_agent = OllamaLocalAgent(base_url="http://localhost:11434", model_name="qwen2.5:7b", verbose=False)

print(f"Generator Agent selected: {generator_agent.__class__.__name__}")
print(f"Judge Agent selected: {judge_agent.__class__.__name__}")

### Step 4: Implement LLM-as-a-Judge Evaluation Logic

In [ ]:
def evaluate_constraints_with_judge(docs, directive, generated_constraints, judge):
    """
    Calls the Judge LLM to evaluate the correctness of must-link and cannot-link constraints
    under the given natural language directive.
    """
    must_link = generated_constraints.get("must_link", [])
    cannot_link = generated_constraints.get("cannot_link", [])
    
    if not must_link and not cannot_link:
        return {"must_link_evaluation": [], "cannot_link_evaluation": [], "accuracy": 1.0}
        
    # Format documents list
    docs_str = ""
    for i, doc in enumerate(docs):
        docs_str += f"--- Document [{i}] ---\n{doc}\n\n"
        
    system_prompt = (
        "You are an objective academic judge evaluating the semantic accuracy of pairwise constraints "
        "generated for a document clustering task.\n\n"
        "The user provided this clustering directive: \"" + directive + "\"\n\n"
        "Here are the 20 documents that were analyzed:\n" + docs_str + "\n"
        "Your task is to verify if each generated constraint makes logical sense under the user's directive:\n"
        "- A Must-Link constraint [i, j] is CORRECT if Document [i] and Document [j] should belong to the SAME cluster according to the directive.\n"
        "- A Cannot-Link constraint [i, j] is CORRECT if Document [i] and Document [j] should belong to DIFFERENT clusters according to the directive.\n\n"
        "You must return ONLY a valid JSON object matching the schema:\n"
        "{\n"
        "  \"must_link_evaluation\": [\n"
        "    {\"pair\": [i, j], \"correct\": true/false, \"reason\": \"brief explanation\"},\n"
        "    ...\n"
        "  ],\n"
        "  \"cannot_link_evaluation\": [\n"
        "    {\"pair\": [i, j], \"correct\": true/false, \"reason\": \"brief explanation\"}\n"
        "  ]\n"
        "}"
    )
    
    user_prompt = (
        "Please evaluate the following generated constraints:\n\n"
        f"Must-Link constraints: {must_link}\n"
        f"Cannot-Link constraints: {cannot_link}\n"
    )
    
    try:
        response_text = judge.generate_text(system_prompt, user_prompt)
        
        from src.agent.cloud_agent import parse_json_from_text
        eval_json = parse_json_from_text(response_text)
        
        ml_evals = eval_json.get("must_link_evaluation", [])
        cl_evals = eval_json.get("cannot_link_evaluation", [])
        
        total_evals = len(ml_evals) + len(cl_evals)
        correct_evals = sum(1 for e in ml_evals if e.get("correct")) + sum(1 for e in cl_evals if e.get("correct"))
        
        accuracy = correct_evals / total_evals if total_evals > 0 else 1.0
        
        return {
            "must_link_evaluation": ml_evals,
            "cannot_link_evaluation": cl_evals,
            "accuracy": accuracy
        }
    except Exception as e:
        print(f"Error during LLM Judge evaluation: {e}")
        return {"must_link_evaluation": [], "cannot_link_evaluation": [], "accuracy": 0.0}

print("Judge evaluation function ready!")

### Step 5: Implement Ground Truth Matching Logic (Level 1)

In [ ]:
def evaluate_against_ground_truth(generated_constraints, sampled_cats):
    """
    Computes Precision, Recall, and F1-score of the generated constraints
    against the actual ground-truth categories of the sampled documents.
    """
    ml_gen = set(tuple(sorted(p)) for p in generated_constraints.get("must_link", []))
    cl_gen = set(tuple(sorted(p)) for p in generated_constraints.get("cannot_link", []))
    
    def get_main_cat(c_str):
        prim = c_str.split()[0]
        return prim.split('.')[0]
        
    main_cats = [get_main_cat(c) for c in sampled_cats]
    
    # Generate ground-truth relations
    ml_gt = set()
    cl_gt = set()
    n = len(sampled_cats)
    for i in range(n):
        for j in range(i + 1, n):
            if main_cats[i] == main_cats[j]:
                ml_gt.add((i, j))
            else:
                cl_gt.add((i, j))
                
    # Calculate Must-Link Metrics
    if ml_gen:
        ml_tp = len(ml_gen.intersection(ml_gt))
        ml_precision = ml_tp / len(ml_gen)
        ml_recall = ml_tp / len(ml_gt) if len(ml_gt) > 0 else 0.0
        ml_f1 = (2 * ml_precision * ml_recall) / (ml_precision + ml_recall) if (ml_precision + ml_recall) > 0 else 0.0
    else:
        ml_precision, ml_recall, ml_f1 = 0.0, 0.0, 0.0
        
    # Calculate Cannot-Link Metrics
    if cl_gen:
        cl_tp = len(cl_gen.intersection(cl_gt))
        cl_precision = cl_tp / len(cl_gen)
        cl_recall = cl_tp / len(cl_gt) if len(cl_gt) > 0 else 0.0
        cl_f1 = (2 * cl_precision * cl_recall) / (cl_precision + cl_recall) if (cl_precision + cl_recall) > 0 else 0.0
    else:
        cl_precision, cl_recall, cl_f1 = 0.0, 0.0, 0.0
        
    # Overall constraint metrics
    total_gen = len(ml_gen) + len(cl_gen)
    total_correct = len(ml_gen.intersection(ml_gt)) + len(cl_gen.intersection(cl_gt))
    gt_accuracy = total_correct / total_gen if total_gen > 0 else 1.0
    
    return {
        "ml_precision": ml_precision,
        "ml_recall": ml_recall,
        "ml_f1": ml_f1,
        "cl_precision": cl_precision,
        "cl_recall": cl_recall,
        "cl_f1": cl_f1,
        "gt_accuracy": gt_accuracy
    }

print("Ground-truth metric calculator ready!")

### Step 6: Run Main Evaluation Loop

In [ ]:
results_data = []

# Mock context representing conversational C3 setup
simulated_history = [
    {"role": "user", "content": "I want to cluster these documents."},
    {"role": "assistant", "content": "What criteria or aspect would you like to use for clustering?"}
]

print("Starting evaluation loop...")
print("=" * 85)

for level, directives in directives_suite.items():
    print(f"\n--- Running Level: {level} ---")
    for directive in directives:
        print(f"\nDirective: \"{directive}\"")
        
        history = simulated_history + [{"role": "user", "content": directive}]
        
        # 1. Generate constraints
        print(" -> Generating constraints...")
        try:
            constraints = generator_agent.generate_constraints(history, sampled_texts)
        except Exception as e:
            print(f"   [Error] Constraint generation failed: {e}")
            constraints = {"must_link": [], "cannot_link": []}
            
        n_ml = len(constraints.get("must_link", []))
        n_cl = len(constraints.get("cannot_link", []))
        print(f"    Generated: {n_ml} must-link, {n_cl} cannot-link constraints.")
        
        # 2. Run LLM Judge Semantic verification
        print(" -> Calling LLM Judge for semantic evaluation...")
        judge_res = evaluate_constraints_with_judge(sampled_texts, directive, constraints, judge_agent)
        semantic_accuracy = judge_res.get("accuracy", 0.0)
        print(f"    LLM Judge Semantic Accuracy: {semantic_accuracy:.2%}")
        
        # 3. Calculate quantitative metrics against ground truth (for Level 1 topic directives)
        gt_metrics = {}
        if "Level 1" in level:
            print(" -> Calculating quantitative metrics against category ground truth...")
            gt_metrics = evaluate_against_ground_truth(constraints, sampled_categories)
            print(f"    GT Accuracy: {gt_metrics.get('gt_accuracy', 0.0):.2%} | ML F1: {gt_metrics.get('ml_f1', 0.0):.4f} | CL F1: {gt_metrics.get('cl_f1', 0.0):.4f}")
            
        res_entry = {
            "level": level,
            "directive": directive,
            "num_must_link": n_ml,
            "num_cannot_link": n_cl,
            "semantic_accuracy": semantic_accuracy,
            "gt_accuracy": gt_metrics.get("gt_accuracy", np.nan),
            "ml_precision": gt_metrics.get("ml_precision", np.nan),
            "ml_recall": gt_metrics.get("ml_recall", np.nan),
            "ml_f1": gt_metrics.get("ml_f1", np.nan),
            "cl_precision": gt_metrics.get("cl_precision", np.nan),
            "cl_recall": gt_metrics.get("cl_recall", np.nan),
            "cl_f1": gt_metrics.get("cl_f1", np.nan)
        }
        results_data.append(res_entry)
        print("-" * 50)

print("\n" + "=" * 85)
print("Evaluation loop completed successfully!")

### Step 7: Tabulate Results

In [ ]:
df_results = pd.DataFrame(results_data)

# Save evaluation results to Output folder
os.makedirs("../Output", exist_ok=True)
df_results.to_json("../Output/rq1_results.json", indent=2, orient="records")
print("Saved evaluation results table to Output/rq1_results.json")

# Display results
pd.set_option('display.max_colwidth', None)
df_results[["level", "directive", "num_must_link", "num_cannot_link", "semantic_accuracy", "gt_accuracy"]]

### Step 8: Analyze & Plot Accuracy Curves

In [ ]:
# Average semantic accuracy by complexity level
level_avgs = df_results.groupby("level")["semantic_accuracy"].mean()

plt.figure(figsize=(10, 5))
colors = ['#4F46E5', '#10B981', '#F59E0B']
level_avgs.plot(kind="bar", color=colors, edgecolor='none', alpha=0.85)

plt.title("RQ1: LLM Judge Semantic Accuracy across Directive Levels", fontsize=14, fontweight='bold', pad=15)
plt.ylabel("Average Semantic Accuracy Score", fontsize=12)
plt.xlabel("Directive Complexity Level", fontsize=12)
plt.ylim(0, 1.05)
plt.xticks(rotation=15, ha="right")
plt.grid(True, axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()

plt.savefig("../Output/rq1_accuracy_chart.png", dpi=150)
plt.show()

print("Average Semantic Accuracy:")
for lvl, val in level_avgs.items():
    print(f" - {lvl}: {val:.2%}")

### Step 9: Plot F1 Scores for Level 1 Directives

In [ ]:
# Extract Level 1 results
df_l1 = df_results[df_results["level"].str.contains("Level 1")].copy()

if not df_l1.empty:
    plt.figure(figsize=(10, 5))
    x = np.arange(len(df_l1))
    width = 0.35

    plt.bar(x - width/2, df_l1["ml_f1"], width, label='Must-Link F1', color='#4F46E5', alpha=0.85)
    plt.bar(x + width/2, df_l1["cl_f1"], width, label='Cannot-Link F1', color='#10B981', alpha=0.85)

    plt.title("RQ1: Quantitative F1 Score against Category Ground Truth", fontsize=14, fontweight='bold', pad=15)
    plt.ylabel("F1 Score", fontsize=12)
    plt.xlabel("Level 1 Directives", fontsize=12)
    plt.xticks(x, [f"Directive {i+1}" for i in range(len(df_l1))])
    plt.ylim(0, 1.05)
    plt.legend(loc="upper right")
    plt.grid(True, axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()
    
    plt.savefig("../Output/rq1_constraint_alignment_chart.png", dpi=150)
    plt.show()
else:
    print("No Level 1 data found to plot constraint alignment.")